# tp-05e アンサンブル（OOF だけでブレンド重み・スタッキングを学習）

**実行場所: ホスト PC（stg）。** tp-05b〜d で保存した OOF 予測（`oof/model_oof/*.parquet`）を合成し、単独の最良モデルより良くなるかを確かめる。

## 方法
| 案 | 内容 |
|---|---|
| mean | 確率の単純平均 |
| logit_mean | logit の単純平均 |
| weighted_logit | logit 空間の非負・合計 1 の重みを LogLoss 最小で学習 |
| greedy | Caruana の前進選択（重複あり） |
| stack_lr | 各モデルの logit を入力にした L2 ロジスティック回帰（較正を兼ねる） |

- p_win・p_top2・p_top3 を**別々に**合成し、最後にレース内の合計を 1・2・3 にそろえる。
- 重みの過剰適合は **leave-one-year-out**（1 年を外して他の年で学習 → 外した年で評価）で見積もる。表の数値はすべてこの方式の「正直な」値。
- 最終的な重みは 2022〜2024 年の OOF 全体で学習し、`ensemble_spec.json` に保存する（tp-06 の test 2025 で一度だけ使う）。
- 2026 年と test 2025 は使わない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import ensemble as en, tp05_data, win_models as wm, zoo_runner

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "ensemble"
OUT.mkdir(parents=True, exist_ok=True)
mlflow.set_experiment("keiba-tp-ensemble")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()

D = tp05_data.load_dataset(WORK)
meta = D["meta"]
PREFER = ["lgb_tuned", "xgb_tuned", "nn_tuned", "lgb_softmax", "linear", "cat_binary", "nn_s", "lgb_binary", "xgb_binary"]
names = [n for n in PREFER if (OOF_DIR / f"{n}.parquet").exists()]
print("合成する候補:", names)
key = ["race_id", "horse_id"]
base = meta[key + ["finish_position", "year"]].merge(pd.read_parquet(OOF_DIR / f"{names[0]}.parquet")[key], on=key, how="inner")
for n in names:
    o = pd.read_parquet(OOF_DIR / f"{n}.parquet")[key + ["p_win", "p_top2", "p_top3"]].rename(columns={f"p_{t}": f"{n}__{t}" for t in ("win", "top2", "top3")})
    base = base.merge(o, on=key, how="inner")
print(f"共通の行 {len(base):,}（{base['year'].min()}〜{base['year'].max()}）")
race = base["race_id"].to_numpy()
years = base["year"].to_numpy()
fin = base["finish_position"].to_numpy()
TOT = {"win": 1.0, "top2": 2.0, "top3": 3.0}
Y = {"win": (fin == 1).astype(float), "top2": (fin <= 2).astype(float), "top3": (fin <= 3).astype(float)}
P = {t: np.stack([base[f"{n}__{t}"].to_numpy() for n in names], axis=1) for t in TOT}

In [ ]:
# 単独モデルと各合成案の LOYO 評価（p_win・p_top2・p_top3 の LogLoss）
rows, LOYO = [], {}
for t in TOT:
    for j, n in enumerate(names):
        rows.append({"target": t, "method": f"single:{n}", "logloss": en.ll(wm.normalize_to_total(race, P[t][:, j], TOT[t]), Y[t])})
    for m in en.METHODS:
        v, out = en.loyo_eval(m, P[t], years, race, Y[t], TOT[t])
        rows.append({"target": t, "method": m, "logloss": v})
        LOYO[(t, m)] = out
R = pd.DataFrame(rows)
piv = R.pivot(index="method", columns="target", values="logloss")[["win", "top2", "top3"]]
display(piv.sort_values("win").round(5))
best_single = {t: R[(R.target == t) & R.method.str.startswith("single:")].sort_values("logloss").iloc[0] for t in TOT}
print({t: (b["method"], round(float(b["logloss"]), 5)) for t, b in best_single.items()})
piv.to_csv(OUT / "loyo_logloss.csv")

In [ ]:
# 各対象で、LOYO の LogLoss が最良の合成案を選ぶ（0.00005 以内なら単純な案を優先）
ORDER = ["mean", "logit_mean", "greedy", "weighted_logit", "stack_lr"]
CHOICE = {}
for t in TOT:
    sub = R[(R.target == t) & R.method.isin(ORDER)].set_index("method")["logloss"]
    best = sub.min()
    CHOICE[t] = next(m for m in ORDER if sub[m] <= best + 5e-5)
print("採用する合成案:", CHOICE)
spec = {"git_commit": GIT_COMMIT, "models": names, "targets": {}}
final_P = {}
for t in TOT:
    m = CHOICE[t]
    Z = en.logit(P[t])
    st = en.fit_method(m, P[t], Z, Y[t])                       # 2022〜2024 全体で学習
    entry = {"method": m}
    if m in ("weighted_logit", "greedy"):
        entry["weights"] = [float(x) for x in st]
    elif m == "stack_lr":
        entry["coef"], entry["intercept"] = [float(x) for x in st.coef_[0]], float(st.intercept_[0])
    spec["targets"][t] = entry
    final_P[t] = LOYO[(t, m)]                                   # 評価は LOYO の予測（正直な値）
    print(f"  {t}: {m} {entry.get('weights') or entry.get('coef') or ''}")
(OUT / "ensemble_spec.json").write_text(json.dumps(spec, ensure_ascii=False, indent=1), encoding="utf-8")

In [ ]:
# 最終アンサンブルの性能（LOYO の予測）と、単独の最良モデルとの差（レース単位の paired bootstrap）
meta_idx = base[["year"]].reset_index(drop=True)
ens = zoo_runner.metrics_by_year(meta_idx, final_P, fin, race, "ensemble")
display(ens.round(5))
best_name = {t: str(best_single[t]["method"]).split(":")[1] for t in TOT}
diffs = []
for t in TOT:
    j = names.index(best_name[t])
    ps = wm.normalize_to_total(race, P[t][:, j], TOT[t])
    b = wm.paired_bootstrap_diff(wm.per_race_loss(race, ps, Y[t]), wm.per_race_loss(race, final_P[t], Y[t]), n_boot=1000)
    diffs.append({"target": t, "best_single": best_name[t], "single_logloss": en.ll(ps, Y[t]), "ensemble_logloss": en.ll(final_P[t], Y[t]), "Δ": b["diff"], "lo": b["lo"], "hi": b["hi"]})
cmp_ = pd.DataFrame(diffs).set_index("target")
display(cmp_.round(5)); cmp_.to_csv(OUT / "ensemble_vs_best_single.csv"); ens.to_csv(OUT / "ensemble_fold_metrics.csv")
oof = base[key + ["year"]].copy()
for t in TOT:
    oof[f"p_{t}"] = final_P[t]
oof.to_parquet(OOF_DIR / "ensemble.parquet", index=False)
with mlflow.start_run(run_name="tp-05e-ensemble", tags={"notebook": "tp-05e", "git_commit": GIT_COMMIT, "models": ",".join(names)}):
    mlflow.log_params({f"method_{t}": CHOICE[t] for t in TOT})
    mlflow.log_metrics({f"loyo_{c}": float(ens[c].mean()) for c in ("logloss_win", "logloss_top2", "logloss_top3", "race_nll", "auc_win", "top1_hit")})
    for t in TOT:
        mlflow.log_metrics({f"single_best_{t}": float(cmp_.loc[t, "single_logloss"]), f"ensemble_{t}": float(cmp_.loc[t, "ensemble_logloss"]), f"delta_{t}": float(cmp_.loc[t, "Δ"])})
    mlflow.log_artifact(str(OUT / "ensemble_spec.json"), "ensemble")
    mlflow.log_artifact(str(OUT / "loyo_logloss.csv"), "ensemble")